# 单一入口：plan、apply、check，然后才 actor

不要在浏览器里追加 Python。Run All 只调用 `scripts/run_actor_session.py`。
check 超时固定 75 秒，不会缩到 29 秒。actor 只在 check 看到 2 张 GPU 且剩余时间不少于 125 秒时启动，ready 45 秒、collective 60 秒、cleanup 20 秒。
命令阶段预算 240 秒。不够就写 `NOT_RUN_BUDGET` 并打包，不当成通过。CPU 上 `actor_admission_passed` 必须是 false。
No persistence：把本文件夹作为 input dataset。新 GPU 会话从头 bootstrap，不要假设上次 pip 还在。不下载权重。


In [ ]:
import subprocess
import sys
from pathlib import Path

def find_root():
    seeds = [Path.cwd(), Path('/kaggle/input'), Path('/kaggle/working')]
    seen = []
    for seed in seeds:
        if seed.exists():
            seen.append(seed)
            seen.extend(seed.glob('*'))
            seen.extend(seed.glob('*/*'))
    for candidate in seen:
        script = candidate / 'scripts' / 'run_actor_session.py'
        if script.is_file():
            return candidate
    raise SystemExit('upload this folder as a Kaggle input dataset')

root = find_root()
raise SystemExit(subprocess.call([sys.executable, str(root / 'scripts' / 'run_actor_session.py'), '--evidence-dir', '/kaggle/working/h3-actor-session', '--budget-s', '240'], cwd=str(root)))
